In [4]:
%pip install anthropic python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
from dotenv import load_dotenv
import anthropic

load_dotenv("../.env")
client = anthropic.Anthropic()

In [6]:
response = client.messages.create(
    model="claude-haiku-5-5",
    max_tokens=200,
    messages=[{"role": "user",
               "content": "Wyjaśnij w jednym zdaniu, czym jest lista sankcji OFAC SDN."}],
)
print(response.content[0].text)
print(response.usage)

Lista SDN (Specially Designated Nationals and Blocked Persons) to prowadzona przez amerykański Urząd Kontroli Aktywów Zagranicznych (OFAC) lista osób, firm i podmiotów objętych sankcjami, z którymi obywatele i firmy USA oraz instytucje finansowe mają zakaz prowadzenia transakcji, a ich majątek w jurysdykcji USA podlega zamrożeniu.
Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=44, output_tokens=158, output_tokens_details=OutputTokensDetails(thinking_tokens=0), server_tool_use=None, service_tier='standard', speed='standard')


In [8]:
import pandas as pd

alerts = pd.read_csv("../data/processed/alerts.csv")
customers = pd.read_csv("../data/processed/customers.csv")
people = pd.read_csv("../data/processed/people.csv")

cust = customers[["customer_id", "dob", "nationality"]].rename(
    columns={"dob": "customer_dob", "nationality": "customer_nationality"})
sdn = people[["id", "name", "dob", "nationality", "passport", "program"]].rename(
    columns={"id": "sdn_id", "name": "sdn_name", "dob": "sdn_dob",
             "nationality": "sdn_nationality", "passport": "sdn_passport"})

cases = alerts.merge(cust, on="customer_id").merge(sdn, on="sdn_id")
cases.head()

,customer_id,customer_name,sdn_id,matched_name,score,label,customer_dob,customer_nationality,sdn_name,sdn_dob,sdn_nationality,sdn_passport,program
0,L00105,"VAKHROMEYEV, Ivan Vasilyevich",40768,"VAKHROMEYEV, Ivan Vasilyevich",100,lookalike,1960-09-06,Poland,"VAKHROMEYEV, Ivan Vasilyevich",29 Dec 1988,Russia,NaN,CYBER2
1,L00199,"KELMENDI, Donata",17312,"KELMENDI, Donata",100,lookalike,2001-05-27,Poland,"KELMENDI, Donata",16 Sep 1992,NaN,NaN,SDNTK
2,L00191,"VERDUGO ARAUJO, Jairo",54671,"VERDUGO ARAUJO, Jairo",100,lookalike,1942-11-16,Germany,"VERDUGO ARAUJO, Jairo",02 Sep 1974,Mexico,NaN,SDGT] [ILLICIT-DRUGS-EO14059
3,L00178,"NGOMA, Willy",46856,"NGOMA, Willy",100,lookalike,1998-02-18,United Kingdom,"NGOMA, Willy",1974,Rwanda,NaN,DRCONGO
4,L00101,"CAZARIN MOLINA, Cesar",49605,"CAZARIN MOLINA, Cesar",100,lookalike,1966-04-27,Germany,"CAZARIN MOLINA, Cesar",06 Feb 1980,Mexico,NaN,ILLICIT-DRUGS-EO14059


In [9]:
cases["expected"] = "FALSE_POSITIVE"
is_true = (cases["label"] == "true_match") & (cases["customer_name"] == cases["sdn_name"])
cases.loc[is_true, "expected"] = "TRUE_MATCH"

sample = pd.concat([
    cases[cases["expected"] == "TRUE_MATCH"].sample(5, random_state=1),
    cases[cases["label"] == "lookalike"].sample(10, random_state=1),
    cases[cases["label"] == "clean"],
]).reset_index(drop=True)

sample["expected"].value_counts()

expected
FALSE_POSITIVE    16
TRUE_MATCH         5
Name: count, dtype: int64

In [10]:
import json

policy = open("../policy/sanctions_policy.md", encoding="utf-8").read()

SYSTEM = (
    "Jesteś analitykiem compliance w banku. Rozstrzygasz alerty sankcyjne "
    "WYŁĄCZNIE według poniższej polityki.\n\n" + policy + "\n\n"
    "Odpowiedz TYLKO obiektem JSON, bez żadnego innego tekstu, w formacie:\n"
    '{"decision": "FALSE_POSITIVE | TRUE_MATCH | ESCALATE", "rule": "R1-R8", '
    '"rationale": "krótkie uzasadnienie po polsku", '
    '"evidence": {"name": "...", "dob": "...", "nationality": "...", "passport": "..."}, '
    '"confidence": 0.0}'
)

In [18]:
def adjudicate(case, model="claude-haiku-5-5"):
    alert_text = (
        f"KLIENT: {case['customer_name']} | DOB: {case['customer_dob']} | "
        f"obywatelstwo: {case['customer_nationality']} | paszport: brak danych\n"
        f"LISTA SDN: {case['sdn_name']} | DOB: {case['sdn_dob']} | "
        f"obywatelstwo: {case['sdn_nationality']} | paszport: {case['sdn_passport']} | "
        f"program: {case['program']}\n"
        f"Podobieństwo nazwiska: {case['score']}/100"
    )
    response = client.messages.create(
        model=model, max_tokens=1500, system=SYSTEM,
        messages=[{"role": "user", "content": alert_text}],
    )
    text = next(b.text for b in response.content if b.type == "text").strip()
    start = text.find("{")
    end = text.rfind("}") + 1
    try:
        return json.loads(text[start:end])
    except json.JSONDecodeError:
        print("Surowa odpowiedź modelu:\n", text)
        raise


In [19]:
case = sample.iloc[0]
print(case[["customer_name", "customer_dob", "sdn_name", "sdn_dob", "expected"]])
adjudicate(case)

customer_name    ARAYA NAVARRO, Jimena Romina
customer_dob                      05 Sep 1983
sdn_name         ARAYA NAVARRO, Jimena Romina
sdn_dob                           05 Sep 1983
expected                           TRUE_MATCH
Name: 0, dtype: object


{'decision': 'TRUE_MATCH',
 'rule': 'R4',
 'rationale': 'Data urodzenia zgodna (05 Sep 1983) oraz obywatelstwo zgodne (Venezuela). Nazwisko i imiona identyczne. Brak numeru paszportu po obu stronach, więc R1 i R2 nie mają zastosowania. Reguła R4 jest spełniona; podobieństwo nazwiska samo w sobie nie uzasadnia decyzji (R7).',
 'evidence': {'name': 'ARAYA NAVARRO, Jimena Romina (klient) = ARAYA NAVARRO, Jimena Romina (SDN)',
  'dob': '05 Sep 1983 (zgodna)',
  'nationality': 'Venezuela (zgodne)',
  'passport': 'brak danych po stronie klienta i listy (nan) - nie porównywalny'},
 'confidence': 0.8}

In [21]:
import time

results = []
for i, case in sample.iterrows():
    start = time.time()
    try:
        out = adjudicate(case)
    except Exception as e:
        out = {"decision": "ERROR", "rule": "", "rationale": str(e), "confidence": 0}
    results.append({
        "customer_id": case["customer_id"],
        "customer_name": case["customer_name"],
        "sdn_name": case["sdn_name"],
        "expected": case["expected"],
        "decision": out["decision"],
        "rule": out.get("rule"),
        "confidence": out.get("confidence"),
        "rationale": out.get("rationale"),
        "seconds": round(time.time() - start, 1),
    })
    print(i, case["customer_name"], "→", out["decision"])

decisions = pd.DataFrame(results)
decisions

0 ARAYA NAVARRO, Jimena Romina → TRUE_MATCH
1 WU, Huihui → TRUE_MATCH
2 LERCH, Gotthard → TRUE_MATCH
3 CHACON MIRANDA, Alejandro → TRUE_MATCH
4 SHEVELIN, Pavel Pavlovich → TRUE_MATCH
5 JONG, Yong Su → FALSE_POSITIVE
6 SHAMLAKH, Alaa → FALSE_POSITIVE
7 RAMSHEH, Ali Reza Nurian → FALSE_POSITIVE
8 TAKEUCHI, Teruaki → FALSE_POSITIVE
9 EISMANT, Natallia Mikalaeuna → FALSE_POSITIVE
10 MOSKALKOVA, Tatyana Nikolaevna → FALSE_POSITIVE
11 RI, Song Chol → FALSE_POSITIVE
12 AL-DAGMA, Aschraf → FALSE_POSITIVE
13 KUZMIN, Mikhail Vladimirovich → FALSE_POSITIVE
14 CAZARIN MOLINA, Cesar → FALSE_POSITIVE
15 Leonid Miguel → ESCALATE
16 Mohammad Almeida → FALSE_POSITIVE
17 Francisco Javier Madrid → FALSE_POSITIVE
18 Armando Linda → FALSE_POSITIVE
19 Mohammed Sommer → FALSE_POSITIVE
20 Ana Taylor → FALSE_POSITIVE


,customer_id,customer_name,sdn_name,expected,decision,rule,confidence,rationale,seconds
0,T00009,"ARAYA NAVARRO, Jimena Romina","ARAYA NAVARRO, Jimena Romina",TRUE_MATCH,TRUE_MATCH,R4,0.85,Data urodzenia jest zgodna co do dnia (05 Sep ...,3.4
1,T00019,"WU, Huihui","WU, Huihui",TRUE_MATCH,TRUE_MATCH,R4,0.85,Data urodzenia zgodna co do dnia (15 Dec 1988)...,3.0
2,T00018,"LERCH, Gotthard","LERCH, Gotthard",TRUE_MATCH,TRUE_MATCH,R4,0.85,Data urodzenia zgodna co do dnia (21 Dec 1942)...,2.9
3,T00010,"CHACON MIRANDA, Alejandro","CHACON MIRANDA, Alejandro",TRUE_MATCH,TRUE_MATCH,R4,0.85,Data urodzenia jest identyczna (22 Jul 1985) i...,3.1
4,T00011,"SHEVELIN, Pavel Pavlovich","SHEVELIN, Pavel Pavlovich",TRUE_MATCH,TRUE_MATCH,R4,0.90,Pełne nazwisko i imiona są zgodne (podobieństw...,3.3
5,L00040,"JONG, Yong Su","JONG, Yong Su",FALSE_POSITIVE,FALSE_POSITIVE,R3,0.95,Rok urodzenia klienta (1955) różni się od roku...,3.1
6,L00065,"SHAMLAKH, Alaa","SHAMLAKH, Alaa",FALSE_POSITIVE,FALSE_POSITIVE,R3,0.95,Rok urodzenia klienta (1953) różni się od roku...,3.2
7,L00077,"RAMSHEH, Ali Reza Nurian","RAMSHEH, Ali Reza Nurian",FALSE_POSITIVE,FALSE_POSITIVE,R3,0.95,Rok urodzenia klienta (1961) różni się od roku...,3.3
8,L00013,"TAKEUCHI, Teruaki","TAKEUCHI, Teruaki",FALSE_POSITIVE,FALSE_POSITIVE,R3,0.95,Rok urodzenia klienta (1963) różni się od roku...,3.7
9,L00144,"EISMANT, Natallia Mikalaeuna","EISMANT, Natallia Mikalaeuna",FALSE_POSITIVE,FALSE_POSITIVE,R3,0.97,Rok urodzenia klienta (1938) różni się od roku...,2.7


In [22]:
decisions["correct"] = decisions["decision"] == decisions["expected"]

print("Skuteczność:", round(decisions["correct"].mean() * 100, 1), "%")
print("Śr. czas decyzji:", round(decisions["seconds"].mean(), 1), "s")
print()
print(pd.crosstab(decisions["expected"], decisions["decision"]))

decisions.to_csv("../data/processed/decisions.csv", index=False)

Skuteczność: 95.2 %
Śr. czas decyzji: 3.2 s

decision        ESCALATE  FALSE_POSITIVE  TRUE_MATCH
expected                                            
FALSE_POSITIVE         1              15           0
TRUE_MATCH             0               0           5


In [24]:
%pip install streamlit


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
!streamlit run ../app/dashboard.py --server.headless true



2026-11-22 15:47:46.104 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://192.168.0.102:8501
  External URL: http://195.88.232.17:8501

  For better performance, install the Watchdog module:

  $ xcode-select --install
  $ pip install watchdog
            
